# Practical 04: From Notebook to Package

**Turning scattered cells into a folder of code somebody else can run**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L09 |
| **Course Outcome** | CO3 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 300 MB |
| **Extra software** | nothing beyond the course environment |
| **Marks** | 10 |

## Aim

1. State why a notebook ceases to be adequate for work others depend on.
2. Convert cells into functions, and functions into a package of modules.
3. Import your own code and call it.
4. Save a trained model and run the project as a command.

## What this practical is about

A notebook holds two things that look alike and are not: the code, and the
state produced by running it in some order. The second is invisible. A variable
defined in a cell that has since been edited remains in memory, so the screen
can display one value while the kernel holds another, and both appear correct.

A **package** removes that ambiguity. Code in a module has no state until it is
imported and called, so running it twice produces the same result or a
reproducible error. Three consequences follow.

- The code can be **imported**, by another file, by a test, or by a service.
- The code can be **run as a command**, which is what a container and a
  pipeline require.
- The code can be **reviewed**, because a file has a diff and a notebook
  largely does not.

The structure chosen here groups code by job rather than by the order it was
written, which is the organising rule for every later practical.

## Before you start

P01 to P03 should be complete. A model can be trained and scored. Familiarity
with functions and with `import` is assumed.

Run the cells from top to bottom. A later cell normally depends on an object
created by an earlier one, so a `NameError` usually indicates a skipped
prerequisite.

If notebook state becomes confusing, restart the kernel, clear all outputs, and
run again from the first cell. In this practical in particular, a confused
kernel is the condition the whole exercise is about.

In [ ]:
# Step 0: check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'joblib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'joblib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

## Step 0b: ensure the shared dataset exists

Every practical in this course uses the same 600 synthetic delivery
observations. The next cell confirms the file is present and reconstructs it
from the course seed if it is absent.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
from pathlib import Path

def find_data(name="delivery_times.csv"):
    """Locate the shared dataset from any working directory.

    Walks upward from the current folder looking for data/<name>, so the
    notebook works whether it is launched from the practical folder, the
    Practicals folder, or the repository root."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in (folder / "data" / name,
                          folder / "Practicals" / "data" / name):
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError(
        f"{name} not found in any data/ folder at or above {here}")

DATA = find_data()
print("dataset:", DATA)


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

# Walkthrough

Read each step, then run its cell.

## Step 1: observe the problem before correcting it

A short demonstration of hidden state. Read both cells before running
them.

In [ ]:
total = 10
total = total + 5
print("total is", total)

Now suppose the first line were changed to `total = 100` and only the second
line re-run. The screen would show `total = 100` at the top and `total is 20`
at the bottom. Both are on screen, and only one is true.

This is not a careless mistake. It is the ordinary way a notebook fails, and it
is the reason production code is not kept in one.

## Step 2: plan the package before writing it

Group code by **job** rather than by the order it happened to be written. This
project has three jobs, so it receives three modules.

```
work/
  delivery/
    __init__.py     marks this folder as a package
    data.py         load the CSV and split it
    features.py     check and describe a single order
    model.py        train, score, save, load
  train.py          the script a person runs
```

One rule governs the division: a module must be describable in one sentence. If
it cannot be, it is doing two jobs and should be split.

In [ ]:
from pathlib import Path

WORK = Path("work")
PKG = WORK / "delivery"
PKG.mkdir(parents=True, exist_ok=True)

print("package folder ready:", PKG.resolve())

## Step 3: write the first module, data.py

This module performs one job: obtain the delivery data and split it into
training and testing partitions.

The file is written from the notebook with `write_text`. In practice it would
be typed in an editor. It is done this way so the whole practical remains in
one place and can be run end to end.

In [ ]:
data_py = '''"""Loading and splitting the delivery dataset."""

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
TARGET = "delivery_min"
SEED = 42


def load_orders(path):
    """Read the delivery CSV into a table."""
    return pd.read_csv(Path(path))


def split_orders(orders, test_size=0.2):
    """Return X_train, X_test, y_train, y_test."""
    X = orders[FEATURES]
    y = orders[TARGET]
    return train_test_split(X, y, test_size=test_size,
                            random_state=SEED)
'''

(PKG / "data.py").write_text(data_py, encoding="utf-8")
print("wrote", PKG / "data.py", f"({len(data_py)} characters)")

Note what moved and what did not. The code is the same code from P02. Only its
address changed, and that change is what makes it importable.

## Step 4: write features.py

One job: judging and describing a single order. No file reading and no model
training appear in it, which is how the single sentence rule is enforced.

In [ ]:
features_py = '''"""Checks and descriptions for one delivery order."""

TRAFFIC_LEVELS = (1, 2, 3)


def minutes_per_km(delivery_min, distance_km):
    """How many minutes each kilometre took."""
    if distance_km <= 0:
        raise ValueError("distance_km must be positive")
    return delivery_min / distance_km


def describe_order(order):
    """A short sentence a human can read."""
    weather = "in the rain" if order["rain"] else "in dry weather"
    return (f"{order['distance_km']} km, "
            f"{order['prep_time_min']} min prep, "
            f"traffic {order['traffic_level']}, {weather}")
'''

(PKG / "features.py").write_text(features_py, encoding="utf-8")
print("wrote", PKG / "features.py")

## Step 5: write model.py

One job: everything concerning the model itself, which is training it, scoring
it, saving it to disk and loading it back.

**joblib** is the standard means of saving a trained scikit-learn model to a
file. Saving matters: retraining on every prediction would make the service
unusable, and would also mean the served model differs from the evaluated
one.

In [ ]:
model_py = '''"""Training, scoring, saving and loading the model."""

import joblib
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error


def train_model(X_train, y_train):
    """Fit a linear regression and hand it back."""
    return LinearRegression().fit(X_train, y_train)


def evaluate(model, X_test, y_test):
    """Mean absolute error, in minutes."""
    return mean_absolute_error(y_test, model.predict(X_test))


def save_model(model, path):
    """Write a trained model to disk."""
    joblib.dump(model, path)
    return path


def load_model(path):
    """Read a trained model back from disk."""
    return joblib.load(path)
'''

(PKG / "model.py").write_text(model_py, encoding="utf-8")
print("wrote", PKG / "model.py")

## Step 6: add __init__.py to make it a package

A folder of `.py` files is a folder. Adding a file named `__init__.py` causes
Python to treat it as a **package** that can be imported.

The file may be empty. The one written here lists what the package offers,
which gives a reader one place to look.

In [ ]:
init_py = '''"""Delivery-time prediction for SCSE3040."""

__version__ = "0.1.0"

from .data import FEATURES, TARGET, load_orders, split_orders
from .model import evaluate, load_model, save_model, train_model

__all__ = [
    "FEATURES", "TARGET", "load_orders", "split_orders",
    "train_model", "evaluate", "save_model", "load_model",
]
'''

(PKG / "__init__.py").write_text(init_py, encoding="utf-8")

for f in sorted(PKG.glob("*.py")):
    print(f"  {f.name:<15} {f.stat().st_size:>5} bytes")

## Step 7: import your own package and use it

Python searches for packages in a list of folders held in `sys.path`. This
package sits inside `work/`, so that folder is added to the list.

The `fresh_import` helper matters. Python caches modules, so after a file is
edited the interpreter must be told to read it again. Every person who writes
modules from a notebook meets this once.

In [ ]:
import importlib
import shutil
import sys

if str(WORK.resolve()) not in sys.path:
    sys.path.insert(0, str(WORK.resolve()))


def fresh_import(name):
    """Import a module, reloading it if it changed on disk.

    The first line deletes Python's compiled cache. Python decides
    whether that cache is stale from the file's size and its
    timestamp to the nearest second -- so an edit that keeps the
    length the same can be missed entirely. Deleting it is simplest.
    """
    shutil.rmtree(PKG / "__pycache__", ignore_errors=True)
    importlib.invalidate_caches()
    module = importlib.import_module(name)
    return importlib.reload(module)


delivery_data = fresh_import("delivery.data")
delivery_model = fresh_import("delivery.model")

orders = delivery_data.load_orders(DATA)
X_train, X_test, y_train, y_test = delivery_data.split_orders(orders)
model = delivery_model.train_model(X_train, y_train)
mae = delivery_model.evaluate(model, X_test, y_test)

print(f"rows {len(orders)}  train {len(X_train)}  test {len(X_test)}")
print(f"MAE {mae:.2f} minutes")

Four lines of your own library performed what a screen of notebook cells
performed in P02, and this version can be run by somebody else, on another
machine, without this notebook.

## Step 8: save the trained model to a file

Training takes time and prediction should not repeat it. The trained model is
saved once and loaded whenever a prediction is required.

In [ ]:
model_path = WORK / "model.joblib"
delivery_model.save_model(model, model_path)

reloaded = delivery_model.load_model(model_path)
same = float(reloaded.predict(X_test.head(1))[0])
original = float(model.predict(X_test.head(1))[0])

print(f"saved to {model_path} "
      f"({model_path.stat().st_size:,} bytes)")
print(f"original model predicts {original:.2f}")
print(f"reloaded model predicts {same:.2f}")
print("identical:", abs(same - original) < 1e-9)

That file is the artifact P07 serves and P08 places in a container. From this
point onward, the phrase the model refers to this file rather than to a
variable in a session.

## Step 9: write the script a person runs

The entry point: an ordinary Python file that a person, a scheduled job, or a
container can execute with one command.

The line `if __name__ == "__main__":` causes the block beneath it to run only
when the file is executed directly, and not when it is imported. Without it,
importing the module would trigger a training run.

In [ ]:
train_py = '''"""Train the delivery-time model and report its error."""

import sys
from pathlib import Path

from delivery import (evaluate, load_orders, save_model,
                      split_orders, train_model)


HERE = Path(__file__).resolve().parent
DEFAULT_DATA = HERE.parent.parent / "data" / "delivery_times.csv"


def main():
    data_path = Path(sys.argv[1]) if len(sys.argv) > 1 else DEFAULT_DATA

    orders = load_orders(data_path)
    X_train, X_test, y_train, y_test = split_orders(orders)
    model = train_model(X_train, y_train)
    mae = evaluate(model, X_test, y_test)

    save_model(model, Path(__file__).parent / "model.joblib")

    print(f"rows: {len(orders)}")
    print(f"MAE: {mae:.2f}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

(WORK / "train.py").write_text(train_py, encoding="utf-8")
print("wrote", WORK / "train.py")

## Step 10: run it as a program

No notebook is involved. This is the command that appears in a Dockerfile in
P08 and in a pipeline step in P11.

In [ ]:
import subprocess

def run_script(name, args=(), cwd=WORK):
    """Run a python file the way a terminal would, and show output."""
    done = subprocess.run(
        [sys.executable, name, *args],
        cwd=cwd, capture_output=True, text=True,
    )
    print(f"$ python {name} {' '.join(args)}".rstrip())
    print(done.stdout.strip() or "(nothing printed)")
    if done.returncode != 0:
        print("STDERR:", done.stderr.strip()[-800:])
    print("exit code:", done.returncode)
    return done

result = run_script("train.py", [str(DATA.resolve())])

Exit code `0` denotes success. Any other value denotes failure, and that number
is precisely what the pipeline in P11 reads to decide whether to continue or
stop. Nothing in the pipeline interprets the text of the output.

# Your turn

The walkthrough is complete. The remaining work is yours.

There are **three tasks**, in increasing order of difficulty, each with a hint.

Where a `# TODO` comment appears, replace that line with your own code. Do not
rename the variable on the left of the `=` sign. The self-check at the end
looks for exactly those names.

When all three have been attempted, run the **self-check** cell. It may be run
as often as required and it never alters the notebook.

## Task T1: add a function to the package

Add a function `average_speed_kmph(distance_km, delivery_min)` to
`delivery/features.py`, returning the average speed in kilometres per hour.

Speed in km/h is `distance_km / (delivery_min / 60)`.

Append it to the existing file rather than replacing the file, then re-import
the module and store the result of `average_speed_kmph(10, 30)` in `T1_speed`.

> **Hint.** Build the new function as a string and use
> `open(PKG / 'features.py', 'a', encoding='utf-8')`, where the `'a'` opens for
> appending. Then `feat = fresh_import('delivery.features')` and call
> `feat.average_speed_kmph(10, 30)`. Ten kilometres in thirty minutes is
> 20 km/h.

In [ ]:
new_function = '''

def average_speed_kmph(distance_km, delivery_min):
    "Average speed of a delivery, in kilometres per hour."
    # TODO: return the speed
    return None
'''

with open(PKG / "features.py", "a", encoding="utf-8") as fh:
    fh.write(new_function)

feat = fresh_import("delivery.features")

# TODO: call the new function with 10 km and 30 minutes
T1_speed = None

print("10 km in 30 minutes =", T1_speed, "km/h")

## Task T2: reject an impossible order

Write a new module `delivery/validate.py` containing one function,
`is_valid_order(order)`, which takes a dictionary and returns `True` only if
all four of the following hold.

- `distance_km` is greater than 0.
- `prep_time_min` is 0 or more.
- `traffic_level` is 1, 2 or 3.
- `rain` is 0 or 1.

Anything else returns `False`. Import it and store the module in
`T2_validate`.

> **Hint.** Build the module as one string and write it with `write_text`,
> exactly as Steps 3 to 5 did. Inside the function, read each value with
> `order["..."]` and `return False` as soon as a rule is broken. Then
> `T2_validate = fresh_import('delivery.validate')`.

In [ ]:
validate_py = '''"Is this order usable?"


def is_valid_order(order):
    "True if the order passes every rule."
    # TODO: write the four checks, return False when one fails
    return None
'''

(PKG / "validate.py").write_text(validate_py, encoding="utf-8")

# TODO: import your new module
T2_validate = None

good = {"distance_km": 5.0, "prep_time_min": 20,
        "traffic_level": 2, "rain": 0}

if T2_validate is None:
    print("Not done yet -- fill in the two TODOs above.")
else:
    print("good order ->", T2_validate.is_valid_order(good))

## Task T3: a second command line script

Write `work/predict.py`: a script that loads the saved model from
`work/model.joblib` and prints one prediction for a 7 km order with 25 minutes
of preparation, traffic level 3, and no rain.

It must print a line beginning with `PREDICTION:` followed by the number of
minutes.

Run it with `run_script("predict.py")` and store the completed process in
`T3_run`.

> **Hint.** Follow the shape of `train.py` from Step 9. Import `load_model`
> from `delivery`, build a one row pandas DataFrame with the four columns, call
> `.predict(...)`, and print `f"PREDICTION: {minutes:.1f}"`. The
> `if __name__ == "__main__":` guard is required.

In [ ]:
predict_py = '''"Predict one delivery, from the command line."

import pandas as pd

from delivery import load_model


def main():
    # TODO: load work/model.joblib, predict one 7 km / 25 min /
    #       traffic 3 / no rain order, and print "PREDICTION: <minutes>"
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

(WORK / "predict.py").write_text(predict_py, encoding="utf-8")

# TODO: run it
T3_run = None

## Self-check

Run the cell below to mark the work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK: run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | delivery.features now offers average_speed_kmph', lambda: hasattr(feat, 'average_speed_kmph'))
_check('T1 | 10 km in 30 minutes is 20 km/h', lambda: abs(float(T1_speed) - 20.0) < 1e-6)
_check('T1 | it still works for another order', lambda: abs(float(feat.average_speed_kmph(6, 45)) - 8.0) < 1e-6)
_check('T2 | delivery/validate.py exists and imports', lambda: (PKG / 'validate.py').is_file() and hasattr(T2_validate, 'is_valid_order'))
_check('T2 | a sensible order is accepted', lambda: T2_validate.is_valid_order({'distance_km': 5.0, 'prep_time_min': 20, 'traffic_level': 2, 'rain': 0}) is True)
_check('T2 | a zero distance is rejected', lambda: T2_validate.is_valid_order({'distance_km': 0, 'prep_time_min': 20, 'traffic_level': 2, 'rain': 0}) is False)
_check('T2 | traffic level 9 and rain 5 are both rejected', lambda: T2_validate.is_valid_order({'distance_km': 5.0, 'prep_time_min': 20, 'traffic_level': 9, 'rain': 0}) is False and T2_validate.is_valid_order({'distance_km': 5.0, 'prep_time_min': 20, 'traffic_level': 2, 'rain': 5}) is False)
_check('T3 | predict.py ran and exited successfully', lambda: T3_run.returncode == 0)
_check('T3 | it printed a PREDICTION line with a sensible number', lambda: 'PREDICTION:' in T3_run.stdout and 20 < float(T3_run.stdout.split('PREDICTION:')[1].split()[0]) < 90)

print("==================================================================")
print("SELF-CHECK   Practical 04: From Notebook to Package")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

## What to submit

1. This notebook, with every cell run and its output visible.
2. The `work/delivery/` folder you built, as a zip.
3. In a markdown cell, one sentence per module stating its single job.

Name the file `P04_<your-roll-number>.ipynb` before uploading it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, package builds and imports | 3 |
| Task T1, a new function added to the package | 2 |
| Task T2, validation logic that rejects invalid orders | 3 |
| Task T3, a script that runs from the command line | 2 |
| **Total** | **10** |

## Read more

- Python, modules and packages: <https://docs.python.org/3/tutorial/modules.html>
- Python, `__main__` and the top level script: <https://docs.python.org/3/library/__main__.html>
- joblib, persistence: <https://joblib.readthedocs.io/en/stable/persistence.html>
- Python packaging user guide: <https://packaging.python.org/en/latest/>